<!-- ISM4543-BANNER -->
# Integrative Assignment on Week 03 Material — ClearPath Health
## Data preparation and feature engineering for patient analytics

**ISM4543 | Applied Data Science | Fall 2026**

| | |
|---|---|
| **Due** | Monday, September 14, 2026, 11:59 PM |
| **Points** | 100, allocated by the six rubric components in the syllabus — Problem Framing 10, Technical Implementation 25, Verification Trail 15, Understanding 25, AI Usage Report 10, Critical Analysis 15. The Parts below are the shape of the work, not separate mark pots. |
| **Submit** | Three deliverables: this notebook (`.ipynb`), your **AI Usage Report** (the full Gemini conversation export *plus* your written account of it), and a **3–5 minute video walkthrough** (hard stop at five minutes). Submit the video as a **file upload on the Canvas assignment** or a **publicly accessible link in the Canvas submission comment** — never embedded in this notebook. **No video, or a link we cannot open, is −15 points.** The reflection section below is part of the notebook. |
| **AI policy** | AI assistance is **expected**, and documenting it is **required**. Explanations and reflections must be **your own words**. |
| **Late work** | 20% deduction up to 24 hours late. **After 24 hours, no credit.** |
| **Grade questions** | Raise any grading issue within **5 days** of the evaluation being posted. After that the evaluation stands. |

> **Why "integrative"?** This assignment does not test this week's technique in
> isolation. It expects you to bring forward the data preparation, evaluation, and
> communication skills from earlier weeks and combine them with what you learned
> this week. Marks are lost for a technically correct model presented without
> framing, verification, or interpretation.

---

## Business Situation

**ClearPath Health** is a regional telemedicine and primary care provider with 18 clinics across the Mid-Atlantic region, serving approximately 120,000 patients. Founded in 2016 as a telehealth startup, ClearPath expanded into brick-and-mortar clinics in 2019. The organization wants to predict **30-day hospital readmission risk** to improve patient outcomes and reduce CMS penalties.

### The Stakeholder Meeting

Dr. Elena Vasquez, Chief Medical Officer, has called a meeting with key stakeholders to discuss the readmission prediction initiative.

---

**Dr. Elena Vasquez (Chief Medical Officer):** "Our 30-day readmission rate is at 14%, well above the 11% national benchmark. CMS penalized us $2.3 million last year. If we can identify high-risk patients *before* discharge, our care coordinators can intervene with follow-up calls, medication reviews, and home visits. But we need a reliable predictive model, and that starts with clean data."

**Raj Mehta (VP Health Informatics):** "Our data lives in three separate systems: the EHR for demographics, a clinical records system for visit-level data stored as nested JSON, and our patient portal for engagement metrics. We've never integrated them for analytics. The join between these systems is going to be messy - not every patient appears in every system."

**Dr. Sarah Kim (Director of Quality Improvement):** "I'm particularly concerned about the missing data patterns. Our older clinics - clinics 1 through 6 - didn't have standardized intake forms, so BMI and smoking status are frequently missing. And our patient satisfaction surveys? Sicker patients are far less likely to respond. That's not random missingness - it's systematic, and if we ignore it, our model will be biased toward healthier patients."

**Marcus Thompson (CFO):** "I've seen suspicious values in our billing data. We have patients with total costs of one cent, negative costs, and a few records showing 4,700 visits in a year - that's clearly a data duplication error. But we also have genuinely high-cost patients with complex conditions spending $80,000 to $150,000 annually. We can't just delete everything above some threshold."

**Dr. Vasquez:** "Here's what I need: take our patient data from all three systems, merge them properly, handle the missing values intelligently, clean up the errors Marcus mentioned, and create features that capture readmission risk factors. Document every decision so we can defend our methodology to CMS auditors."

---

## Problem Definition

### The Core Challenge

Prepare ClearPath Health's patient data from three separate systems for a 30-day readmission prediction model by:
1. Loading and merging data from CSV, nested JSON, and CSV sources
2. Systematically handling missing values with appropriate strategies for each missingness type
3. Detecting and treating outliers (data errors vs. genuine extremes)
4. Engineering features that capture readmission risk factors
5. Creating a clean, documented dataset that prevents data leakage

### Business Requirements

| Requirement | Rationale |
|-------------|----------|
| **Preserve patient volume** | Don't delete rows unnecessarily; every patient matters for training |
| **Document all decisions** | Must explain methodology to CMS auditors |
| **Create clinical features** | Care coordinators need interpretable risk indicators |
| **Prevent data leakage** | Model must be valid for deployment in discharge workflow |
| **Handle three data sources** | Merge CSV, nested JSON, and engagement data correctly |

### Success Criteria

1. All three data sources loaded and merged into a single analysis-ready dataset
2. No missing values in the final dataset
3. Outliers addressed with documented rationale (errors vs. genuine extremes)
4. At least 5 engineered features with clinical justification
5. Proper train-test split with no data leakage

---

## Part 1: Problem Framing
Before writing any code, demonstrate your understanding of the business problem.

### 1.1 Problem Statement

**In your own words, what is the core data preparation challenge ClearPath Health faces? Why can't they just delete rows with missing data?**

*Your answer (2-3 sentences): ClearPath Health needs to combine and clean data from different sources while dealing with missing values, incorrect records, and different levels of detail. They cannot simply delete rows with missing data because this could remove many important patients and make the final dataset less representative of the real patient population.*


### 1.2 Data Quality Impact

**Dr. Kim warned that satisfaction scores are missing in a systematic pattern (sicker patients don't respond). Explain how this type of missingness (MNAR) could bias a readmission prediction model if handled incorrectly. Give a specific example.**

*Your answer (3-4 sentences): If sicker patients are less likely to complete satisfaction surveys, the missing values are not random. If we simply delete those patients or fill their missing scores without considering this pattern, the model may learn mostly from healthier patients. For example, patients with missing satisfaction scores may actually have a higher chance of being readmitted, but deleting them could make the model underestimate readmission risk for high-risk patients.*


### 1.3 Key Metrics

**How will you measure the success of your data preparation? List 2-3 specific metrics or checks you'll use.**

*Your answer: I will measure the success of the data preparation by checking:*

* *The percentage of missing values before and after cleaning.*
* *Whether invalid or unrealistic values have been corrected while keeping as many patients as possible.*
* *Whether the training and testing datasets have similar readmission rates and no missing values after preprocessing.*


---

## Part 2: Data Loading and Initial Exploration

Run the cell below to load the ClearPath Health patient data from three separate systems.

In [1]:
import pandas as pd
import numpy as np
import json, urllib.request
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
import warnings
warnings.filterwarnings('ignore')

# --- Load ClearPath Health data from three systems ---
BASE_URL = "https://raw.githubusercontent.com/prof-tcsmith/4543Fall26-DATA/main/data/week03/assignment"

# System 1: Patient demographics (CSV)
patients = pd.read_csv(f"{BASE_URL}/patients.csv")

# System 2: Clinical visit records (nested JSON → flatten)
with urllib.request.urlopen(f"{BASE_URL}/clinical_records.json") as resp:
    clinical_raw = json.load(resp)
clinical = pd.json_normalize(clinical_raw)

# System 3: Patient engagement metrics (CSV)
engagement = pd.read_csv(f"{BASE_URL}/patient_engagement.csv")

print("=" * 60)
print("ClearPath Health Data Loaded Successfully")
print("=" * 60)
print(f"\nPatients (demographics):    {patients.shape[0]:,} rows × {patients.shape[1]} columns")
print(f"Clinical records (visits):  {clinical.shape[0]:,} rows × {clinical.shape[1]} columns")
print(f"Patient engagement:         {engagement.shape[0]:,} rows × {engagement.shape[1]} columns")
print(f"\nMissing values exist: {patients.isnull().any().any() or clinical.isnull().any().any() or engagement.isnull().any().any()}")

ClearPath Health Data Loaded Successfully

Patients (demographics):    800 rows × 10 columns
Clinical records (visits):  2,507 rows × 13 columns
Patient engagement:         800 rows × 10 columns

Missing values exist: True


### 2.1 Initial Data Exploration

**Task:** Explore each of the three datasets. Your exploration should include:
- Data types and basic statistics for each table
- Missing value counts and percentages
- How the three tables relate to each other (join key, cardinality)

*Document your prompts in the AI Usage Report section at the end.*

In [2]:
# Your code here - Initial exploration of all three datasets

print("=" * 60)
print("Initial Data Exploration for ClearPath Health DataFrames")
print("=" * 60)

# --- Patients DataFrame Exploration ---
print("\n" + "="*50 + "\nPatients DataFrame Exploration\n" + "="*50)
print("\nShape:", patients.shape)
print("\nFirst 5 rows:\n", patients.head())
print("\nInfo (Data Types & Non-Null Counts):\n")
patients.info()
print("\nBasic Statistics (Numerical Columns):\n", patients.describe())

print("\nMissing Values (Count and Percentage):")
missing_patients = patients.isnull().sum()
missing_patients_percent = (patients.isnull().sum() / len(patients)) * 100
missing_patients_df = pd.DataFrame({
    'Missing Count': missing_patients,
    'Missing Percentage': missing_patients_percent
})
print(
    missing_patients_df[
        missing_patients_df['Missing Count'] > 0
    ].sort_values(
        by='Missing Count',
        ascending=False
    )
)

print("\nDuplicate Rows (Full row duplication):", patients.duplicated().sum())
print("\nUnique patient_id count:", patients['patient_id'].nunique())
print(
    "Is patient_id unique in patients DataFrame?",
    patients['patient_id'].nunique() == len(patients)
)

# --- Clinical DataFrame Exploration ---
print("\n" + "="*50 + "\nClinical DataFrame Exploration\n" + "="*50)
print("\nShape:", clinical.shape)
print("\nFirst 5 rows:\n", clinical.head())
print("\nInfo (Data Types & Non-Null Counts):\n")
clinical.info()
print("\nBasic Statistics (Numerical Columns):\n", clinical.describe())

print("\nMissing Values (Count and Percentage):")
missing_clinical = clinical.isnull().sum()
missing_clinical_percent = (clinical.isnull().sum() / len(clinical)) * 100
missing_clinical_df = pd.DataFrame({
    'Missing Count': missing_clinical,
    'Missing Percentage': missing_clinical_percent
})
print(
    missing_clinical_df[
        missing_clinical_df['Missing Count'] > 0
    ].sort_values(
        by='Missing Count',
        ascending=False
    )
)

print("\nDuplicate Rows (Full row duplication):", clinical.duplicated().sum())
print("\nUnique patient_id count:", clinical['patient_id'].nunique())
print(
    "Is patient_id unique in clinical DataFrame?",
    clinical['patient_id'].nunique() == len(clinical)
)

print("\nNumber of clinical records per patient:")
clinical_records_per_patient = clinical['patient_id'].value_counts()

print(
    f"  Total unique patients in clinical records: "
    f"{clinical_records_per_patient.shape[0]}"
)
print(
    f"  Min records per patient: "
    f"{clinical_records_per_patient.min()}"
)
print(
    f"  Max records per patient: "
    f"{clinical_records_per_patient.max()}"
)
print(
    f"  Mean records per patient: "
    f"{clinical_records_per_patient.mean():.2f}"
)
print(
    f"  Median records per patient: "
    f"{clinical_records_per_patient.median()}"
)
print(
    f"  Standard deviation of records per patient: "
    f"{clinical_records_per_patient.std():.2f}"
)
print(
    "  Top 5 patients by number of records:\n",
    clinical_records_per_patient.head()
)

# --- Engagement DataFrame Exploration ---
print("\n" + "="*50 + "\nEngagement DataFrame Exploration\n" + "="*50)
print("\nShape:", engagement.shape)
print("\nFirst 5 rows:\n", engagement.head())
print("\nInfo (Data Types & Non-Null Counts):\n")
engagement.info()
print("\nBasic Statistics (Numerical Columns):\n", engagement.describe())

print("\nMissing Values (Count and Percentage):")
missing_engagement = engagement.isnull().sum()
missing_engagement_percent = (engagement.isnull().sum() / len(engagement)) * 100
missing_engagement_df = pd.DataFrame({
    'Missing Count': missing_engagement,
    'Missing Percentage': missing_engagement_percent
})
print(
    missing_engagement_df[
        missing_engagement_df['Missing Count'] > 0
    ].sort_values(
        by='Missing Count',
        ascending=False
    )
)

print("\nDuplicate Rows (Full row duplication):", engagement.duplicated().sum())
print("\nUnique patient_id count:", engagement['patient_id'].nunique())
print(
    "Is patient_id unique in engagement DataFrame?",
    engagement['patient_id'].nunique() == len(engagement)
)

# --- Table Relationships ---
print("\n" + "="*50 + "\nTable Relationships\n" + "="*50)
print("\nCommon Join Key: 'patient_id'")

patients_ids = set(patients['patient_id'].unique())
clinical_ids = set(clinical['patient_id'].unique())
engagement_ids = set(engagement['patient_id'].unique())

print(
    f"\nNumber of unique patient_ids in 'patients': "
    f"{len(patients_ids)}"
)
print(
    f"Number of unique patient_ids in 'clinical': "
    f"{len(clinical_ids)}"
)
print(
    f"Number of unique patient_ids in 'engagement': "
    f"{len(engagement_ids)}"
)

print(
    f"\nPatients present in all three datasets: "
    f"{len(patients_ids.intersection(clinical_ids, engagement_ids))}"
)
print(
    f"Patients in 'patients' only: "
    f"{len(patients_ids - (clinical_ids | engagement_ids))}"
)
print(
    f"Patients in 'clinical' only: "
    f"{len(clinical_ids - (patients_ids | engagement_ids))}"
)
print(
    f"Patients in 'engagement' only: "
    f"{len(engagement_ids - (patients_ids | clinical_ids))}"
)

print("\nRelationship between 'patients' and 'engagement':")
print(
    f"  Both DataFrames have {len(patients_ids)} unique patient_ids."
)
print(
    f"  Are all patient_ids in 'patients' also in 'engagement'? "
    f"{patients_ids.issubset(engagement_ids)}"
)
print(
    f"  Are all patient_ids in 'engagement' also in 'patients'? "
    f"{engagement_ids.issubset(patients_ids)}"
)
print(
    "  Cardinality: One-to-one because patient_id is unique in both "
    "datasets and the patient_id sets are identical."
)

print("\nRelationship between 'patients' (or 'engagement') and 'clinical':")
print(
    "  'patients' and 'engagement' contain patient-level data, "
    "while 'clinical' contains visit-level data."
)
print(
    "  This means one patient can have multiple clinical records."
)
print(
    f"  All patient_ids from 'patients' are present in 'clinical'? "
    f"{patients_ids.issubset(clinical_ids)}"
)
print(
    f"  All patient_ids from 'clinical' are present in 'patients'? "
    f"{clinical_ids.issubset(patients_ids)}"
)
print(
    "  Cardinality: One-to-many from 'patients' or 'engagement' "
    "to 'clinical', using patient_id as the join key."
)

clinical_only = len(clinical_ids - patients_ids)

print(f"  Clinical-only patient IDs: {clinical_only}")

if (
    patients_ids == clinical_ids
    and patients_ids == engagement_ids
):
    print(
        "  All 800 patient IDs are represented in all three datasets."
    )
else:
    print(
        "  Some patient IDs are not represented in all three datasets."
    )

Initial Data Exploration for ClearPath Health DataFrames

Patients DataFrame Exploration

Shape: (800, 10)

First 5 rows:
   patient_id  age  gender   bmi smoking_status insurance_type  clinic_id  \
0    CP00001   62  Female  20.2        Current        Private         17   
1    CP00002   41  Female  20.0          Never        Private          8   
2    CP00003   70    Male  27.5          Never       Self-Pay         10   
3    CP00004   73  Female  24.1         Former       Medicare         14   
4    CP00005    0  Female  30.8         Former       Medicare         13   

  enrollment_channel enrollment_date  zip_code  
0          In-Person      2018-09-26     20010  
1          In-Person      2017-05-21     21201  
2            Website      2025-09-23     20002  
3            Website      2018-12-07     22201  
4          In-Person      2017-02-04     22030  

Info (Data Types & Non-Null Counts):

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (t

### 2.2 Merge the Three Sources

**Task:** Merge the three datasets into a single DataFrame for analysis.

**Consider carefully:**
- What join type should you use? (inner, left, outer?) Why?
- The clinical records have multiple rows per patient (one per visit). How will you handle this?
- Which columns become the join keys?

In [3]:
# Your code here - Merge the three datasets

print("=" * 60)
print("Merging ClearPath Health DataFrames")
print("=" * 60)

# Aggregate clinical visits to one row per patient
clinical_agg = clinical.groupby("patient_id").agg(
    num_clinical_visits=("visit_id", "count"),
    avg_num_medications=("num_medications", "mean"),
    max_num_medications=("num_medications", "max"),
    avg_num_chronic_conditions=("num_chronic_conditions", "mean"),
    max_num_chronic_conditions=("num_chronic_conditions", "max"),
    was_readmitted_30d=("was_readmitted_30d", "max"),
    most_common_diagnosis_category=(
        "diagnosis.category",
        lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
    ),
    has_chronic_diagnosis=("diagnosis.chronic", "max"),
    avg_systolic_bp=("vitals.systolic_bp", "mean"),
    avg_diastolic_bp=("vitals.diastolic_bp", "mean"),
    avg_heart_rate=("vitals.heart_rate", "mean")
).reset_index()

print(f"\nClinical shape before aggregation: {clinical.shape}")
print(f"Clinical shape after aggregation: {clinical_agg.shape}")

# Merge patient demographics with engagement data
patients_engagement = pd.merge(
    patients,
    engagement,
    on="patient_id",
    how="left",
    validate="one_to_one"
)

print(f"Patients + Engagement shape: {patients_engagement.shape}")

# Merge with patient-level clinical data
merged_df = pd.merge(
    patients_engagement,
    clinical_agg,
    on="patient_id",
    how="left",
    validate="one_to_one"
)

print(f"Final merged shape: {merged_df.shape}")

# Verification
print("\nVerification Checks")
print("-" * 40)
print("Unique patient IDs:", merged_df["patient_id"].nunique())
print("Duplicate patient IDs:", merged_df["patient_id"].duplicated().sum())

unmatched_engagement = set(patients["patient_id"]) - set(engagement["patient_id"])
unmatched_clinical = set(patients["patient_id"]) - set(clinical_agg["patient_id"])

print("Patients missing from engagement:", len(unmatched_engagement))
print("Patients missing from clinical:", len(unmatched_clinical))

print("\nReadmission target distribution:")
print(merged_df["was_readmitted_30d"].value_counts())

print("\nFirst 5 rows:")
display(merged_df.head())

Merging ClearPath Health DataFrames

Clinical shape before aggregation: (2507, 13)
Clinical shape after aggregation: (800, 12)
Patients + Engagement shape: (800, 19)
Final merged shape: (800, 30)

Verification Checks
----------------------------------------
Unique patient IDs: 800
Duplicate patient IDs: 0
Patients missing from engagement: 0
Patients missing from clinical: 0

Readmission target distribution:
was_readmitted_30d
False    485
True     315
Name: count, dtype: int64

First 5 rows:


,patient_id,age,gender,bmi,smoking_status,insurance_type,clinic_id,enrollment_channel,enrollment_date,zip_code,...,avg_num_medications,max_num_medications,avg_num_chronic_conditions,max_num_chronic_conditions,was_readmitted_30d,most_common_diagnosis_category,has_chronic_diagnosis,avg_systolic_bp,avg_diastolic_bp,avg_heart_rate
0,CP00001,62,Female,20.2,Current,Private,17,In-Person,2018-09-26,20010,...,3.333333,4,2.666667,4,True,Cardiovascular,True,128.000000,74.333333,79.333333
1,CP00002,41,Female,20.0,Never,Private,8,In-Person,2017-05-21,21201,...,4.000000,4,3.000000,3,False,Respiratory,True,150.000000,79.000000,75.000000
2,CP00003,70,Male,27.5,Never,Self-Pay,10,Website,2025-09-23,20002,...,4.666667,5,2.333333,4,False,Endocrine,True,137.500000,87.000000,61.000000
3,CP00004,73,Female,24.1,Former,Medicare,14,Website,2018-12-07,22201,...,2.750000,3,2.500000,4,False,Cardiovascular,True,125.750000,77.500000,80.750000
4,CP00005,0,Female,30.8,Former,Medicare,13,In-Person,2017-02-04,22030,...,5.333333,7,3.666667,7,True,Cardiovascular,True,138.666667,78.000000,79.000000


**Merge Decisions:**

Based on initial data exploration, all 800 unique patients appear in the `patients`, `clinical`, and `engagement` datasets. The `patients` and `engagement` DataFrames have a one-to-one relationship with `patient_id`.

The `clinical` DataFrame, however, had a one-to-many relationship with `patient_id` (multiple visits per patient). To merge this with the patient-level data, `clinical` was aggregated to one row per `patient_id`. Sensible aggregations were used for clinical metrics (e.g., mean/max for medications and chronic conditions, count for visits, mode for diagnosis category). The `was_readmitted_30d` target was set to `True` if a patient had *any* readmission across their visits (using the `max` aggregation).

To preserve all 800 patients as required, `left merges` were used: first merging `patients` with `engagement`, and then merging the result with the aggregated `clinical` data, all on `patient_id`. This merge strategy ensured no patients were dropped. The final merged DataFrame, `merged_df`, has a shape of (800, 30), with one unique row per patient, ready for further analysis.

---

## Part 3: Missing Value Analysis and Treatment
### 3.1 Analyze Missing Value Patterns

**Task:** Determine the type of missingness (MCAR, MAR, MNAR) for each column with missing values. Use summary statistics and cross-tabulations to support your analysis.

**Hint:** Dr. Kim mentioned two patterns. Older clinics (1-6) have different missing patterns than newer clinics — check that. She also believes satisfaction scores are missing more for sicker patients — **check that too, rather than assuming it.** A claim from a stakeholder is a hypothesis, not a finding. Investigate these patterns.

In [4]:
import pandas as pd
import numpy as np

print("=" * 60)
print("3.1 Missing Value Analysis")
print("=" * 60)

# Overall missing values
missing_info = pd.DataFrame({
    "Missing Count": merged_df.isna().sum(),
    "Missing %": (merged_df.isna().mean() * 100).round(2)
})

missing_info = missing_info[missing_info["Missing Count"] > 0]

print("\nOverall Missing Values:")
display(missing_info.sort_values("Missing %", ascending=False))

# --------------------------------------------------
# Clinic-related missingness
# --------------------------------------------------
old_clinic = merged_df["clinic_id"].between(1, 6)

clinic_cols = [
    "bmi",
    "smoking_status",
    "insurance_type",
    "avg_systolic_bp",
    "avg_diastolic_bp",
    "avg_heart_rate"
]

clinic_results = []

for col in clinic_cols:
    old_pct = merged_df.loc[old_clinic, col].isna().mean() * 100
    new_pct = merged_df.loc[~old_clinic, col].isna().mean() * 100

    clinic_results.append({
        "Column": col,
        "Clinics 1-6 Missing %": round(old_pct, 2),
        "Clinics 7-18 Missing %": round(new_pct, 2)
    })

print("\nClinic Missingness Comparison:")
clinic_compare = pd.DataFrame(clinic_results)
display(clinic_compare)

# --------------------------------------------------
# Satisfaction score missingness
# --------------------------------------------------
sat_missing = merged_df["satisfaction_score"].isna()

sickness_cols = [
    "num_clinical_visits",
    "avg_num_medications",
    "max_num_chronic_conditions",
    "total_cost_12mo",
    "ed_visits_12mo",
    "avg_systolic_bp"
]

sat_compare = pd.DataFrame({
    "Missing Satisfaction": merged_df.loc[
        sat_missing, sickness_cols
    ].mean(),
    "Present Satisfaction": merged_df.loc[
        ~sat_missing, sickness_cols
    ].mean()
}).round(2)

print("\nSatisfaction Missingness vs. Sickness Indicators:")
display(sat_compare)

readmission_missing = (
    merged_df.loc[sat_missing, "was_readmitted_30d"].mean() * 100
)

readmission_present = (
    merged_df.loc[~sat_missing, "was_readmitted_30d"].mean() * 100
)

print(
    f"Readmission rate - Missing satisfaction: "
    f"{readmission_missing:.1f}%"
)
print(
    f"Readmission rate - Present satisfaction: "
    f"{readmission_present:.1f}%"
)

# --------------------------------------------------
# Evidence table
# --------------------------------------------------
def miss_pct(col):
    return f"{merged_df[col].isna().mean() * 100:.2f}%"

bmi_old = (
    merged_df.loc[old_clinic, "bmi"].isna().mean() * 100
)
bmi_new = (
    merged_df.loc[~old_clinic, "bmi"].isna().mean() * 100
)

smoke_old = (
    merged_df.loc[old_clinic, "smoking_status"].isna().mean() * 100
)
smoke_new = (
    merged_df.loc[~old_clinic, "smoking_status"].isna().mean() * 100
)

evidence_df = pd.DataFrame([
    [
        "satisfaction_score",
        miss_pct("satisfaction_score"),
        "Unclear / Possible MNAR",
        f"Mixed evidence. Readmission was {readmission_missing:.1f}% "
        f"vs {readmission_present:.1f}%. Some burden measures were "
        "slightly higher, but the pattern was not consistent."
    ],
    [
        "app_usage_monthly",
        miss_pct("app_usage_monthly"),
        "Unclear",
        "No clear missingness pattern was identified."
    ],
    [
        "bmi",
        miss_pct("bmi"),
        "MAR",
        f"Missingness was {bmi_old:.1f}% in clinics 1-6 "
        f"and {bmi_new:.1f}% in clinics 7-18."
    ],
    [
        "smoking_status",
        miss_pct("smoking_status"),
        "MAR",
        f"Missingness was {smoke_old:.1f}% in clinics 1-6 "
        f"and {smoke_new:.1f}% in clinics 7-18."
    ],
    [
        "insurance_type",
        miss_pct("insurance_type"),
        "Possibly MCAR / Unclear",
        "Missingness was similar between older and newer clinics."
    ],
    [
        "messages_to_provider_90d",
        miss_pct("messages_to_provider_90d"),
        "Unclear",
        "No clear missingness pattern was identified."
    ],
    [
        "avg_systolic_bp",
        miss_pct("avg_systolic_bp"),
        "Possibly MCAR / Unclear",
        "Missingness was similar between older and newer clinics."
    ],
    [
        "avg_diastolic_bp",
        miss_pct("avg_diastolic_bp"),
        "Possibly MCAR / Unclear",
        "Missingness was similar between older and newer clinics."
    ],
    [
        "avg_heart_rate",
        miss_pct("avg_heart_rate"),
        "Possibly MCAR / Unclear",
        "Missingness was similar between older and newer clinics."
    ]
], columns=[
    "Column",
    "Missing %",
    "Likely Type",
    "Evidence"
])

print("\nMissingness Type Evidence:")
display(evidence_df)

3.1 Missing Value Analysis

Overall Missing Values:


,Missing Count,Missing %
satisfaction_score,321,40.12
app_usage_monthly,164,20.50
bmi,144,18.00
smoking_status,144,18.00
insurance_type,40,5.00
messages_to_provider_90d,40,5.00
avg_systolic_bp,17,2.12
avg_diastolic_bp,17,2.12
avg_heart_rate,17,2.12



Clinic Missingness Comparison:


,Column,Clinics 1-6 Missing %,Clinics 7-18 Missing %
0,bmi,54.96,0.00
1,smoking_status,54.96,0.00
2,insurance_type,5.73,4.65
3,avg_systolic_bp,1.91,2.23
4,avg_diastolic_bp,1.91,2.23
5,avg_heart_rate,1.91,2.23



Satisfaction Missingness vs. Sickness Indicators:


,Missing Satisfaction,Present Satisfaction
num_clinical_visits,3.11,3.15
avg_num_medications,4.03,4.14
max_num_chronic_conditions,4.03,3.95
total_cost_12mo,8524.49,8367.02
ed_visits_12mo,0.93,0.77
avg_systolic_bp,129.18,131.26


Readmission rate - Missing satisfaction: 40.2%
Readmission rate - Present satisfaction: 38.8%

Missingness Type Evidence:


,Column,Missing %,Likely Type,Evidence
0,satisfaction_score,40.12%,Unclear / Possible MNAR,Mixed evidence. Readmission was 40.2% vs 38.8%...
1,app_usage_monthly,20.50%,Unclear,No clear missingness pattern was identified.
2,bmi,18.00%,MAR,Missingness was 55.0% in clinics 1-6 and 0.0% ...
3,smoking_status,18.00%,MAR,Missingness was 55.0% in clinics 1-6 and 0.0% ...
4,insurance_type,5.00%,Possibly MCAR / Unclear,Missingness was similar between older and newe...
5,messages_to_provider_90d,5.00%,Unclear,No clear missingness pattern was identified.
6,avg_systolic_bp,2.12%,Possibly MCAR / Unclear,Missingness was similar between older and newe...
7,avg_diastolic_bp,2.12%,Possibly MCAR / Unclear,Missingness was similar between older and newe...
8,avg_heart_rate,2.12%,Possibly MCAR / Unclear,Missingness was similar between older and newe...


**Missing Value Pattern Analysis:**

| Column | Missing % | Likely Type | Evidence |
|---|---:|---|---|
| satisfaction_score | 40.12% | Unclear / Possible MNAR | Missing satisfaction scores show mixed evidence of higher patient burden. Readmission is slightly higher for missing scores (40.2% vs 38.8%), and chronic conditions and total cost are also slightly higher. However, visits, medications, and systolic BP are not higher, so Dr. Kim’s MNAR hypothesis is possible but not proven. |
| app_usage_monthly | 20.50% | Unclear | No clear pattern was identified from the current analysis, so there is not enough evidence to classify it as MCAR, MAR, or MNAR. |
| bmi | 18.00% | MAR | Missingness is 55.0% in clinics 1–6 and 0.0% in newer clinics, showing that missingness is strongly related to the observed variable `clinic_id`. |
| smoking_status | 18.00% | MAR | Missingness is 55.0% in clinics 1–6 and 0.0% in newer clinics, showing that missingness is strongly related to the observed variable `clinic_id`. |
| insurance_type | 5.00% | Possibly MCAR / Unclear | Missingness is similar in older clinics (5.7%) and newer clinics (4.6%), so clinic type does not appear to explain the missingness. More evidence would be needed to confirm MCAR. |
| messages_to_provider_90d | 5.00% | Unclear | No clear pattern was identified from the current analysis, so more investigation would be needed. |
| avg_systolic_bp | 2.12% | Possibly MCAR / Unclear | Missingness is similar in older clinics (1.9%) and newer clinics (2.2%), so clinic type does not appear to explain the missingness. |
| avg_diastolic_bp | 2.12% | Possibly MCAR / Unclear | Missingness is similar in older clinics (1.9%) and newer clinics (2.2%), so clinic type does not appear to explain the missingness. |
| avg_heart_rate | 2.12% | Possibly MCAR / Unclear | Missingness is similar in older clinics (1.9%) and newer clinics (2.2%), so clinic type does not appear to explain the missingness. |

### 3.2 Compare Imputation Strategies

**Task:** For numeric columns with missing values, compare at least two imputation strategies:
1. Simple imputation (mean or median)
2. KNN imputation

Evaluate which preserves the data distribution better using summary statistics (mean, median, std, quartiles).

In [5]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("3.2 Comparing Imputation Strategies for Numeric Columns")
print("=" * 60)

# Numeric predictors for KNN
exclude_cols = ['clinic_id', 'zip_code', 'was_readmitted_30d']

knn_features = [
    col for col in merged_df.select_dtypes(include=np.number).columns
    if col not in exclude_cols
]

# Numeric columns that actually contain missing values
numeric_missing_cols = [
    col for col in knn_features
    if merged_df[col].isnull().any()
]

print("\nNumeric columns with missing values:")
print(numeric_missing_cols)

# Function for descriptive statistics
def get_stats(series):
    return pd.Series({
        'Mean': series.mean(),
        'Median': series.median(),
        'Std': series.std(),
        'Q1': series.quantile(0.25),
        'Q3': series.quantile(0.75)
    }).round(2)

# --------------------------------------------------
# KNN imputation: scale first, fit once, then restore scale
# --------------------------------------------------
knn_data = merged_df[knn_features].copy()

scaler = StandardScaler()
scaled_data = scaler.fit_transform(knn_data)

knn_imputer = KNNImputer(n_neighbors=5)
knn_scaled_imputed = knn_imputer.fit_transform(scaled_data)

knn_imputed_data = pd.DataFrame(
    scaler.inverse_transform(knn_scaled_imputed),
    columns=knn_features,
    index=merged_df.index
)

# --------------------------------------------------
# Compare Median vs KNN
# --------------------------------------------------
summary_results = []

for col in numeric_missing_cols:

    print(f"\n--- Analyzing column: '{col}' ---")

    # Original observed distribution
    original_stats = get_stats(merged_df[col])

    # Median imputation
    median_imputed = merged_df[col].fillna(merged_df[col].median())
    median_stats = get_stats(median_imputed)

    # KNN imputation
    knn_stats = get_stats(knn_imputed_data[col])

    comparison_df = pd.DataFrame({
        'Original': original_stats,
        'Median Imputed': median_stats,
        'KNN Imputed': knn_stats
    })

    display(comparison_df)

    # Compare how much each method changes the original statistics
    median_difference = abs(original_stats - median_stats).sum()
    knn_difference = abs(original_stats - knn_stats).sum()

    if knn_difference < median_difference:
        best_method = "KNN"
    elif median_difference < knn_difference:
        best_method = "Median"
    else:
        best_method = "Similar"

    summary_results.append({
        'Column': col,
        'Median Difference': round(median_difference, 2),
        'KNN Difference': round(knn_difference, 2),
        'Better Distribution Preservation': best_method
    })

# --------------------------------------------------
# Final summary
# --------------------------------------------------
print("\n" + "=" * 60)
print("Overall Imputation Comparison")
print("=" * 60)

summary_df = pd.DataFrame(summary_results)
display(summary_df)

3.2 Comparing Imputation Strategies for Numeric Columns

Numeric columns with missing values:
['bmi', 'app_usage_monthly', 'messages_to_provider_90d', 'satisfaction_score', 'avg_systolic_bp', 'avg_diastolic_bp', 'avg_heart_rate']

--- Analyzing column: 'bmi' ---


,Original,Median Imputed,KNN Imputed
Mean,28.24,28.26,28.30
Median,28.35,28.35,28.50
Std,5.95,5.39,5.54
Q1,24.10,25.30,24.83
Q3,32.20,31.40,31.80



--- Analyzing column: 'app_usage_monthly' ---


,Original,Median Imputed,KNN Imputed
Mean,8.00,8.00,7.99
Median,8.00,8.00,8.00
Std,2.77,2.47,2.53
Q1,6.00,7.00,6.40
Q3,10.00,9.00,9.05



--- Analyzing column: 'messages_to_provider_90d' ---


...

,Original,Median Imputed,KNN Imputed
Mean,79.62,79.62,79.62
Median,79.50,79.50,79.50
Std,6.49,6.42,6.44
Q1,75.37,75.62,75.50
Q3,83.50,83.50,83.50



--- Analyzing column: 'avg_heart_rate' ---


,Original,Median Imputed,KNN Imputed
Mean,77.39,77.39,77.42
Median,77.50,77.50,77.50
Std,8.00,7.92,7.95
Q1,72.50,72.65,72.50
Q3,82.00,82.00,82.00



Overall Imputation Comparison


,Column,Median Difference,KNN Difference,Better Distribution Preservation
0,bmi,2.58,1.75,KNN
1,app_usage_monthly,2.30,1.60,KNN
2,messages_to_provider_90d,1.07,1.04,KNN
3,satisfaction_score,1.12,0.61,KNN
4,avg_systolic_bp,0.27,0.34,Median
5,avg_diastolic_bp,0.32,0.18,KNN
6,avg_heart_rate,0.23,0.08,KNN


### 3.3 Implement Final Imputation Strategy

**Task:** Based on your analysis, implement the appropriate imputation strategy for each column. Handle both numeric and categorical missing values.

**Remember:** Consider creating missing indicator columns before imputing. They cost one column and preserve the fact that a value was absent — which matters most when you are *not sure* the missingness is random.

In [6]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("3.3 Final Imputation Strategy")
print("=" * 60)

# Keep merged_df unchanged
imputed_df = merged_df.copy()

# --------------------------------------------------
# 1. Missing indicators
# --------------------------------------------------
indicator_cols = ['bmi', 'smoking_status', 'satisfaction_score']

for col in indicator_cols:
    imputed_df[f'{col}_was_missing'] = imputed_df[col].isna().astype(int)

print("\nMissing indicators created for:")
print(indicator_cols)

# --------------------------------------------------
# 2. Missing values before treatment
# --------------------------------------------------
print("\nMissing Values BEFORE Imputation:")
missing_before = imputed_df.isna().sum()
print(missing_before[missing_before > 0])

# --------------------------------------------------
# 3. Categorical imputation
# --------------------------------------------------
categorical_cols = ['smoking_status', 'insurance_type']

for col in categorical_cols:
    if imputed_df[col].isna().any():
        mode_value = imputed_df[col].mode()[0]
        imputed_df[col] = imputed_df[col].fillna(mode_value)

# --------------------------------------------------
# 4. KNN numeric imputation
# --------------------------------------------------
knn_cols = [
    'bmi',
    'app_usage_monthly',
    'messages_to_provider_90d',
    'satisfaction_score',
    'avg_diastolic_bp'
]

# Numeric predictors used for KNN
exclude_cols = [
    'clinic_id',
    'zip_code',
    'was_readmitted_30d'
]

knn_predictors = [
    col for col in imputed_df.select_dtypes(include=np.number).columns
    if col not in exclude_cols
    and not col.endswith('_was_missing')
]

knn_data = imputed_df[knn_predictors].copy()

# Scale before KNN
scaler = StandardScaler()
scaled_data = scaler.fit_transform(knn_data)

knn_imputer = KNNImputer(n_neighbors=5)
scaled_imputed = knn_imputer.fit_transform(scaled_data)

knn_imputed_df = pd.DataFrame(
    scaler.inverse_transform(scaled_imputed),
    columns=knn_predictors,
    index=imputed_df.index
)

for col in knn_cols:
    imputed_df[col] = knn_imputed_df[col]

# --------------------------------------------------
# 5. Median imputation
# --------------------------------------------------
median_cols = [
    'avg_systolic_bp',
    'avg_heart_rate'
]

for col in median_cols:
    imputed_df[col] = imputed_df[col].fillna(
        imputed_df[col].median()
    )

# --------------------------------------------------
# 6. Restore sensible count values
# --------------------------------------------------
count_cols = [
    'app_usage_monthly',
    'messages_to_provider_90d'
]

for col in count_cols:
    imputed_df[col] = imputed_df[col].round().astype('Int64')

# --------------------------------------------------
# 7. Verification
# --------------------------------------------------
print("\nMissing Values AFTER Imputation:")

missing_after = imputed_df.isna().sum()
remaining_missing = missing_after[missing_after > 0]

if remaining_missing.empty:
    print("No missing values remain.")
else:
    print(remaining_missing)

print("\nMissing indicator totals:")
for col in indicator_cols:
    indicator = f'{col}_was_missing'
    print(f"{indicator}: {imputed_df[indicator].sum()}")

print("\nFinal imputed dataset shape:", imputed_df.shape)

3.3 Final Imputation Strategy

Missing indicators created for:
['bmi', 'smoking_status', 'satisfaction_score']

Missing Values BEFORE Imputation:
bmi                         144
smoking_status              144
insurance_type               40
app_usage_monthly           164
messages_to_provider_90d     40
satisfaction_score          321
avg_systolic_bp              17
avg_diastolic_bp             17
avg_heart_rate               17
dtype: int64

Missing Values AFTER Imputation:
No missing values remain.

Missing indicator totals:
bmi_was_missing: 144
smoking_status_was_missing: 144
satisfaction_score_was_missing: 321

Final imputed dataset shape: (800, 33)


**Imputation Strategy Justification:**

| Column                                                               | Strategy Chosen              | Clinical Rationale                                                                                                          |
| -------------------------------------------------------------------- | ---------------------------- | --------------------------------------------------------------------------------------------------------------------------- |
| `bmi`, `app_usage_monthly`, `satisfaction_score`, `avg_diastolic_bp` | KNN imputation               | KNN preserved the original distributions better than median imputation and uses information from similar patients.          |
| `avg_systolic_bp`, `avg_heart_rate`                                  | Median imputation            | Only a small number of values were missing, and median imputation kept the distributions very close to the original data.   |
| `smoking_status`, `insurance_type`                                   | Mode imputation              | These are categorical variables, so the most common category is a simple and appropriate replacement.                       |
| `messages_to_provider_90d`                                           | KNN imputation, then rounded | KNN slightly preserved the distribution better, and the values were rounded because message counts should be whole numbers. |
| Missing indicators for `bmi`, `smoking_status`, `satisfaction_score` | Added before imputation      | Their missingness may contain useful information, so the model can still know that the original value was missing.          |


---

## Part 4: Outlier Detection and Treatment

### 4.1 Identify Outliers

**Task:** Use the IQR method and summary statistics to identify outliers. Distinguish between:
- **Data entry errors** (impossible values — e.g., age of 200, systolic BP of 320)
- **Genuine extreme values** (legitimate but unusual — e.g., high-cost complex patients)

**Hint:** Marcus mentioned billing errors (costs of $0.01 and negative values), duplicate-record errors (4,700 visits), and impossible vitals. But genuinely high-cost patients ($80K-$150K) and genuinely hypertensive patients (systolic 180-195) are real.

In [7]:
import pandas as pd
import numpy as np

print("=" * 60)
print("4.1 Outlier Detection")
print("=" * 60)

# Numeric variables for IQR review
numeric_cols = [
    col for col in imputed_df.select_dtypes(include=np.number).columns
    if col not in ["clinic_id", "zip_code"]
    and not col.endswith("_was_missing")
]

# --------------------------------------------------
# IQR outlier counts
# --------------------------------------------------
iqr_results = []

for col in numeric_cols:
    q1 = imputed_df[col].quantile(0.25)
    q3 = imputed_df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = (
        (imputed_df[col] < lower) |
        (imputed_df[col] > upper)
    ).sum()

    iqr_results.append({
        "Column": col,
        "IQR Outliers": int(count),
        "Min": round(imputed_df[col].min(), 2),
        "Max": round(imputed_df[col].max(), 2)
    })

print("\nIQR Outlier Summary:")
display(pd.DataFrame(iqr_results))

# --------------------------------------------------
# Business-rule checks for clear data errors
# --------------------------------------------------
age_errors = imputed_df[
    (imputed_df["age"] <= 0) |
    (imputed_df["age"] > 120)
]

visit_errors = imputed_df[
    imputed_df["total_visits_12mo"] >= 1000
]

cost_errors = imputed_df[
    imputed_df["total_cost_12mo"] <= 0.01
]

systolic_errors = imputed_df[
    (imputed_df["avg_systolic_bp"] < 50) |
    (imputed_df["avg_systolic_bp"] > 250)
]

# Genuine extremes to preserve
high_cost_patients = imputed_df[
    imputed_df["total_cost_12mo"].between(80000, 150000)
]

high_bp_patients = imputed_df[
    imputed_df["avg_systolic_bp"].between(180, 195)
]

classification = pd.DataFrame([
    {
        "Column": "age",
        "Outliers Found": len(age_errors),
        "Observed Range":
            f"{imputed_df['age'].min()} to "
            f"{imputed_df['age'].max()}",
        "Type": "Data-entry error"
    },
    {
        "Column": "total_visits_12mo",
        "Outliers Found": len(visit_errors),
        "Observed Range":
            f"{imputed_df['total_visits_12mo'].min()} to "
            f"{imputed_df['total_visits_12mo'].max()}",
        "Type": "Data-entry / duplication error"
    },
    {
        "Column": "total_cost_12mo",
        "Outliers Found": len(cost_errors),
        "Observed Range":
            f"${imputed_df['total_cost_12mo'].min():,.2f} to "
            f"${imputed_df['total_cost_12mo'].max():,.2f}",
        "Type": "Invalid low values; high-cost cases may be genuine"
    },
    {
        "Column": "avg_systolic_bp",
        "Outliers Found": len(systolic_errors),
        "Observed Range":
            f"{imputed_df['avg_systolic_bp'].min():.1f} to "
            f"{imputed_df['avg_systolic_bp'].max():.1f}",
        "Type": "Impossible values are errors; plausible highs kept"
    }
])

print("\nKey Outlier Classification:")
display(classification)

# --------------------------------------------------
# Suspicious rows
# --------------------------------------------------
suspicious_ids = set(
    age_errors["patient_id"].tolist()
    + visit_errors["patient_id"].tolist()
    + cost_errors["patient_id"].tolist()
    + systolic_errors["patient_id"].tolist()
)

print("\nLikely Data-Entry Errors:")

if suspicious_ids:
    display(
        imputed_df[
            imputed_df["patient_id"].isin(suspicious_ids)
        ][[
            "patient_id",
            "age",
            "total_visits_12mo",
            "total_cost_12mo",
            "avg_systolic_bp"
        ]]
    )
else:
    print("No obvious data-entry errors found.")

print("\nGenuine Extremes to Preserve:")
print(
    "High-cost patients ($80K-$150K):",
    len(high_cost_patients)
)
print(
    "Average systolic BP 180-195:",
    len(high_bp_patients)
)

4.1 Outlier Detection

IQR Outlier Summary:


,Column,IQR Outliers,Min,Max
0,age,10,-3.0,250.0
1,bmi,8,16.0,46.0
2,portal_logins_monthly,0,0.0,14.0
3,app_usage_monthly,18,1.0,17.0
4,messages_to_provider_90d,10,0.0,9.0
5,appointment_adherence_pct,4,24.9,100.0
6,satisfaction_score,22,1.0,5.0
7,days_since_last_visit,37,1.0,365.0
8,total_visits_12mo,12,1.0,4700.0
9,ed_visits_12mo,40,0.0,5.0



Key Outlier Classification:


,Column,Outliers Found,Observed Range,Type
0,age,10,-3 to 250,Data-entry error
1,total_visits_12mo,2,1.0 to 4700.0,Data-entry / duplication error
2,total_cost_12mo,4,"$-500.00 to $149,630.00",Invalid low values; high-cost cases may be gen...
3,avg_systolic_bp,1,85.0 to 258.0,Impossible values are errors; plausible highs ...



Likely Data-Entry Errors:


,patient_id,age,total_visits_12mo,total_cost_12mo,avg_systolic_bp
4,CP00005,0,3.0,428.69,138.666667
128,CP00129,47,4.0,-500.00,122.600000
253,CP00254,42,5.0,0.01,122.000000
256,CP00257,64,3200.0,2961.54,132.333333
275,CP00276,64,2.0,4017.64,258.000000
319,CP00320,195,7.0,26989.80,172.000000
359,CP00360,185,6.0,44426.11,139.500000
362,CP00363,73,3.0,-250.00,129.500000
422,CP00423,200,6.0,7128.81,135.666667
444,CP00445,52,4700.0,5817.03,127.000000



Genuine Extremes to Preserve:
High-cost patients ($80K-$150K): 6
Average systolic BP 180-195: 0


### 4.2 Treat Outliers

**Task:** For each identified outlier category, decide on treatment:
- **Remove/Replace:** Clear data entry errors (impossible ages, negative costs)
- **Cap (Winsorize):** Legitimate extreme values
- **Keep:** If the outlier represents important clinical information

In [8]:
import numpy as np
import pandas as pd

print("=" * 60)
print("4.2 Outlier Treatment")
print("=" * 60)

# Keep imputed_df unchanged
cleaned_df = imputed_df.copy()

rules = {
    "age": (
        (cleaned_df["age"] <= 0) |
        (cleaned_df["age"] > 120),
        "Impossible ages at or below 0 or above 120 were replaced."
    ),

    "total_cost_12mo": (
        cleaned_df["total_cost_12mo"] <= 0.01,
        "Negative or $0.01 costs were replaced; valid high costs kept."
    ),

    "total_visits_12mo": (
        cleaned_df["total_visits_12mo"] >= 1000,
        "Thousands-level visit counts were treated as recording errors."
    ),

    "avg_systolic_bp": (
        (cleaned_df["avg_systolic_bp"] < 50) |
        (cleaned_df["avg_systolic_bp"] > 250),
        "Impossible systolic BP values were replaced; plausible highs kept."
    )
}

treatment_results = []

for col, (condition, rationale) in rules.items():

    before_min = cleaned_df[col].min()
    before_max = cleaned_df[col].max()

    n_treated = int(condition.sum())

    # Replace clear errors with NaN
    cleaned_df.loc[condition, col] = np.nan

    # Median calculated from valid remaining values
    median_value = cleaned_df[col].median()
    cleaned_df[col] = cleaned_df[col].fillna(median_value)

    after_min = cleaned_df[col].min()
    after_max = cleaned_df[col].max()

    treatment_results.append({
        "Column": col,
        "Outliers Found": n_treated,
        "Treatment Applied":
            "Replace with NaN + median imputation",
        "Before Range":
            f"{before_min:.2f} to {before_max:.2f}",
        "After Range":
            f"{after_min:.2f} to {after_max:.2f}",
        "Clinical Rationale": rationale
    })

print("\nPatient count before treatment:", len(imputed_df))
print("Patient count after treatment:", len(cleaned_df))
print("Patients removed:", len(imputed_df) - len(cleaned_df))

print("\nOutlier Treatment Decisions:")
treatment_summary_df = pd.DataFrame(treatment_results)
display(treatment_summary_df)

print("\nGenuine Extremes Preserved:")
print(
    "High-cost patients ($80K-$150K):",
    cleaned_df["total_cost_12mo"]
    .between(80000, 150000)
    .sum()
)

print(
    "Average systolic BP 180-195:",
    cleaned_df["avg_systolic_bp"]
    .between(180, 195)
    .sum()
)

4.2 Outlier Treatment

Patient count before treatment: 800
Patient count after treatment: 800
Patients removed: 0

Outlier Treatment Decisions:


,Column,Outliers Found,Treatment Applied,Before Range,After Range,Clinical Rationale
0,age,10,Replace with NaN + median imputation,-3.00 to 250.00,22.00 to 92.00,Impossible ages at or below 0 or above 120 wer...
1,total_cost_12mo,4,Replace with NaN + median imputation,-500.00 to 149630.00,162.15 to 149630.00,Negative or $0.01 costs were replaced; valid h...
2,total_visits_12mo,2,Replace with NaN + median imputation,1.00 to 4700.00,1.00 to 13.00,Thousands-level visit counts were treated as r...
3,avg_systolic_bp,1,Replace with NaN + median imputation,85.00 to 258.00,85.00 to 179.00,Impossible systolic BP values were replaced; p...



Genuine Extremes Preserved:
High-cost patients ($80K-$150K): 6
Average systolic BP 180-195: 0


**Outlier Treatment Decisions**

| Column              | Outliers Found | Treatment Applied         | Clinical Rationale                                                                                                      |
| ------------------- | -------------: | ------------------------- | ----------------------------------------------------------------------------------------------------------------------- |
| `age`               |              8 | Replace with NaN + median | Impossible ages below 0 or above 120 were treated as data-entry errors.                                                 |
| `total_cost_12mo`   |              4 | Replace with NaN + median | Negative or $0.01 costs are invalid billing values, while legitimate $80K–$150K costs were kept.                        |
| `total_visits_12mo` |              2 | Replace with NaN + median | Thousands of visits in one year were treated as duplicate/recording errors; the valid maximum became 13.                |
| `avg_systolic_bp`   |              1 | Replace with NaN + median | The impossible value above 250 was corrected, while clinically plausible high BP values were not automatically removed. |


---

## Part 5: Feature Engineering

### 5.1 Create Derived Features

**Task:** Create at least 5 new features that would be valuable for predicting 30-day readmission. Consider:
- **Clinical risk indicators** (e.g., chronic condition burden, medication complexity)
- **Engagement signals** (e.g., portal activity, appointment adherence)
- **Utilization patterns** (e.g., ED visit rate, cost per visit)
- **Time-based features** (e.g., enrollment tenure, recency of last visit)

Each feature must have a clear clinical rationale for why it might predict readmission.

In [9]:
import pandas as pd
import numpy as np

print("=" * 60)
print("5.1 Feature Engineering")
print("=" * 60)

features_df = cleaned_df.copy()

# Convert enrollment date
features_df["enrollment_date"] = pd.to_datetime(
    features_df["enrollment_date"]
)

# --------------------------------------------------
# 1. Chronic condition burden
# --------------------------------------------------
features_df["Chronic_Condition_Burden"] = (
    features_df["avg_num_chronic_conditions"]
    + features_df["has_chronic_diagnosis"].astype(int)
)

# --------------------------------------------------
# 2. Medication complexity
# --------------------------------------------------
features_df["Medication_Complexity_Index"] = (
    features_df["avg_num_medications"]
    + features_df["max_num_medications"]
)

# --------------------------------------------------
# 3. Relative ED utilization
# --------------------------------------------------
features_df["ED_Visits_Ratio"] = np.where(
    features_df["total_visits_12mo"] > 0,
    features_df["ed_visits_12mo"]
    / features_df["total_visits_12mo"],
    0
)

# --------------------------------------------------
# 4. Cost per visit
# --------------------------------------------------
features_df["Cost_Per_Visit"] = np.where(
    features_df["total_visits_12mo"] > 0,
    features_df["total_cost_12mo"]
    / features_df["total_visits_12mo"],
    0
)

# --------------------------------------------------
# 5. Engagement level
# --------------------------------------------------
features_df["Engagement_Level"] = (
    features_df["portal_logins_monthly"]
    + features_df["app_usage_monthly"]
    + features_df["messages_to_provider_90d"]
)

# --------------------------------------------------
# 6. Enrollment tenure
# --------------------------------------------------
reference_date = pd.Timestamp("2025-12-31")

features_df["Enrollment_Tenure_Months"] = (
    (
        reference_date
        - features_df["enrollment_date"]
    ).dt.days / 30.44
).clip(lower=0).round(2)

new_features = [
    "Chronic_Condition_Burden",
    "Medication_Complexity_Index",
    "ED_Visits_Ratio",
    "Cost_Per_Visit",
    "Engagement_Level",
    "Enrollment_Tenure_Months"
]

print("\nSummary Statistics for New Features:")
display(features_df[new_features].describe().round(2))

print("\nMissing values in engineered features:")
print(features_df[new_features].isna().sum())

print("\nInfinite values in engineered features:")
print(
    np.isinf(
        features_df[new_features]
    ).sum()
)

print("\nFeature engineering completed.")

5.1 Feature Engineering

Summary Statistics for New Features:


,Chronic_Condition_Burden,Medication_Complexity_Index,ED_Visits_Ratio,Cost_Per_Visit,Engagement_Level,Enrollment_Tenure_Months
count,800.00,800.00,800.00,800.00,800.0,800.00
mean,3.35,9.66,0.24,2291.81,16.49,60.53
std,1.30,3.20,0.39,4778.18,3.73,34.30
min,0.00,0.00,0.00,40.01,6.0,0.07
25%,2.60,7.50,0.00,512.50,14.0,31.60
50%,3.33,9.78,0.14,1134.86,16.0,61.27
75%,4.00,11.67,0.33,2399.74,19.0,88.78
max,12.00,24.00,4.00,98606.00,28.0,119.94



Missing values in engineered features:
Chronic_Condition_Burden       0
Medication_Complexity_Index    0
ED_Visits_Ratio                0
Cost_Per_Visit                 0
Engagement_Level               0
Enrollment_Tenure_Months       0
dtype: int64

Infinite values in engineered features:
Chronic_Condition_Burden       0
Medication_Complexity_Index    0
ED_Visits_Ratio                0
Cost_Per_Visit                 0
Engagement_Level               0
Enrollment_Tenure_Months       0
dtype: Int64

Feature engineering completed.


**Engineered Features Documentation:**

**Engineered Features Documentation:**

| Feature Name | Formula / Logic | Clinical Rationale | Expected Predictive Value |
|---|---|---|---|
| `Chronic_Condition_Burden` | Average chronic conditions + chronic diagnosis indicator | Patients with more chronic conditions usually need more complex care. | Higher values may increase readmission risk. |
| `Medication_Complexity_Index` | Average medications + maximum medications | More medications may show more complex medical management. | Higher values may increase readmission risk. |
| `ED_Visits_Ratio` | ED visits / total visits | Shows relative use of emergency care compared with overall care. | Higher values may indicate greater instability and readmission risk. |
| `Cost_Per_Visit` | Total annual cost / total annual visits | Higher cost per visit may reflect more intensive or complex care. | Higher values may increase readmission risk. |
| `Engagement_Level` | Portal logins + app usage + provider messages | More engagement may show better communication and care management. | Higher engagement may reduce readmission risk. |
| `Enrollment_Tenure_Months` | 2025-12-31 minus enrollment date, converted to months | Longer enrollment may reflect stronger continuity of care. | Longer tenure may be related to lower readmission risk. |

### 5.2 Encode Categorical Variables

**Task:** Implement appropriate encoding for all categorical variables:
- One-hot encoding for nominal variables (e.g., `gender`, `insurance_type`)
- Ordinal encoding for ordered categories if any exist
- Consider cardinality and the dummy variable trap

In [10]:
import pandas as pd

print("=" * 60)
print("5.2 Encoding Categorical Variables")
print("=" * 60)

# Working copy
encoded_df = features_df.copy()

# Raw date is no longer needed because tenure was engineered
encoded_df = encoded_df.drop(columns=['enrollment_date'])

# Treat clinic ID and ZIP code as categorical, not continuous numbers
encoded_df['clinic_id'] = encoded_df['clinic_id'].astype(str)
encoded_df['zip_code'] = encoded_df['zip_code'].astype(str)

# Identify categorical predictors
categorical_cols = encoded_df.select_dtypes(include='object').columns.tolist()

# Exclude patient identifier
categorical_cols = [
    col for col in categorical_cols
    if col != 'patient_id'
]

print("\nCategorical variables and cardinality:")
for col in categorical_cols:
    print(f"{col}: {encoded_df[col].nunique()} categories")

print("\nShape before encoding:", encoded_df.shape)

# One-hot encode nominal variables
encoded_df = pd.get_dummies(
    encoded_df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

# Convert remaining boolean predictors/target to 0/1
bool_cols = encoded_df.select_dtypes(include='bool').columns
encoded_df[bool_cols] = encoded_df[bool_cols].astype(int)

print("Shape after encoding:", encoded_df.shape)

print("\nRemaining object columns:")
print(encoded_df.select_dtypes(include='object').columns.tolist())

print("\nEncoding completed.")
display(encoded_df.head())

5.2 Encoding Categorical Variables

Categorical variables and cardinality:
gender: 3 categories
smoking_status: 3 categories
insurance_type: 4 categories
clinic_id: 18 categories
enrollment_channel: 4 categories
zip_code: 12 categories
most_common_diagnosis_category: 6 categories

Shape before encoding: (800, 38)
Shape after encoding: (800, 74)

Remaining object columns:
['patient_id']

Encoding completed.


,patient_id,age,bmi,portal_logins_monthly,app_usage_monthly,messages_to_provider_90d,appointment_adherence_pct,satisfaction_score,days_since_last_visit,total_visits_12mo,...,zip_code_21202,zip_code_21218,zip_code_22030,zip_code_22101,zip_code_22201,most_common_diagnosis_category_Endocrine,most_common_diagnosis_category_Mental Health,most_common_diagnosis_category_Musculoskeletal,most_common_diagnosis_category_Neurological,most_common_diagnosis_category_Respiratory
0,CP00001,62.0,20.2,7.0,6,5,76.1,3.44,93,5.0,...,0,0,0,0,0,0,0,0,0,0
1,CP00002,41.0,20.0,4.0,9,5,83.0,4.30,36,4.0,...,0,0,0,0,0,0,0,0,0,1
2,CP00003,70.0,27.5,9.0,7,2,69.6,3.62,26,2.0,...,0,0,0,0,0,1,0,0,0,0
3,CP00004,73.0,24.1,4.0,6,3,73.0,4.20,21,3.0,...,0,0,0,0,1,0,0,0,0,0
4,CP00005,58.0,30.8,6.0,11,1,61.5,3.80,12,3.0,...,0,0,1,0,0,0,0,0,0,0


### 5.3 Feature Selection

**Task:** Evaluate your features and remove any that are:
- Highly correlated with others (>0.85 correlation)
- Low variance (near-constant values)
- Potential sources of data leakage

In [11]:
import pandas as pd
import numpy as np

print("=" * 60)
print("5.3 Feature Selection")
print("=" * 60)

model_df = encoded_df.copy()
removed_features = []

# --------------------------------------------------
# 1. Remove patient identifier
# --------------------------------------------------
if "patient_id" in model_df.columns:
    model_df = model_df.drop(columns=["patient_id"])

    removed_features.append({
        "Feature Removed": "patient_id",
        "Reason": "Identifier",
        "Method Used": "Manual review"
    })

# Separate target
target = "was_readmitted_30d"

y = model_df[target].astype(int)
X = model_df.drop(columns=[target])

print("Initial number of features:", X.shape[1])

# --------------------------------------------------
# 2. Constant / near-constant check
# --------------------------------------------------
constant_cols = [
    col for col in X.columns
    if X[col].nunique() <= 1
]

near_constant_cols = []

for col in X.columns:
    if X[col].nunique() > 1:
        dominant_share = (
            X[col].value_counts(normalize=True, dropna=False)
            .iloc[0]
        )

        if dominant_share >= 0.99:
            near_constant_cols.append(col)

low_variance_cols = list(
    dict.fromkeys(
        constant_cols + near_constant_cols
    )
)

if low_variance_cols:
    X = X.drop(columns=low_variance_cols)

    for col in low_variance_cols:
        removed_features.append({
            "Feature Removed": col,
            "Reason": "Constant or near-constant feature",
            "Method Used": "Low-variance check"
        })

print(
    "Low-variance features removed:",
    low_variance_cols if low_variance_cols else "None"
)

# --------------------------------------------------
# 3. Exact duplicate columns
# --------------------------------------------------
duplicate_cols = []

columns = X.columns.tolist()

for i, col1 in enumerate(columns):
    for col2 in columns[i + 1:]:
        if col2 in X.columns and X[col1].equals(X[col2]):
            duplicate_cols.append(col2)

duplicate_cols = list(dict.fromkeys(duplicate_cols))

if duplicate_cols:
    X = X.drop(columns=duplicate_cols)

    for col in duplicate_cols:
        removed_features.append({
            "Feature Removed": col,
            "Reason": "Duplicate information",
            "Method Used": "Exact column comparison"
        })

# --------------------------------------------------
# 4. High correlation check
# --------------------------------------------------
corr_matrix = X.corr(numeric_only=True).abs()

high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):

        value = corr_matrix.iloc[i, j]

        if value > 0.85:
            high_corr_pairs.append({
                "Feature 1": corr_matrix.columns[i],
                "Feature 2": corr_matrix.columns[j],
                "Correlation": round(value, 3)
            })

print("\nHighly Correlated Feature Pairs (>0.85):")

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs)
    display(high_corr_df)
else:
    print("None found.")

# --------------------------------------------------
# 5. Remove redundant engineered features
# --------------------------------------------------
redundant_candidates = [
    "Medication_Complexity_Index",
    "Chronic_Condition_Burden"
]

for col in redundant_candidates:

    related = [
        pair for pair in high_corr_pairs
        if col in [
            pair["Feature 1"],
            pair["Feature 2"]
        ]
    ]

    if col in X.columns and related:

        X = X.drop(columns=[col])

        removed_features.append({
            "Feature Removed": col,
            "Reason":
                "Highly correlated with an existing clinical feature",
            "Method Used": "Correlation > 0.85"
        })

# --------------------------------------------------
# 6. Leakage verification
# --------------------------------------------------
print("\nLeakage Check:")
print(
    "Target included in X:",
    target in X.columns
)
print(
    "patient_id included in X:",
    "patient_id" in X.columns
)

# --------------------------------------------------
# Summary
# --------------------------------------------------
print("\n" + "=" * 60)
print("Feature Selection Decisions")
print("=" * 60)

removed_df = pd.DataFrame(removed_features)

if not removed_df.empty:
    display(removed_df)
else:
    print("No features removed.")

print("\nFinal number of features:", X.shape[1])

print("\nFinal Feature Set:")
print(X.columns.tolist())

5.3 Feature Selection
Initial number of features: 72
Low-variance features removed: None

Highly Correlated Feature Pairs (>0.85):


,Feature 1,Feature 2,Correlation
0,avg_num_medications,Medication_Complexity_Index,0.918
1,max_num_medications,Medication_Complexity_Index,0.963
2,avg_num_chronic_conditions,Chronic_Condition_Burden,0.940



Leakage Check:
Target included in X: False
patient_id included in X: False

Feature Selection Decisions


,Feature Removed,Reason,Method Used
0,patient_id,Identifier,Manual review
1,Medication_Complexity_Index,Highly correlated with an existing clinical fe...,Correlation > 0.85
2,Chronic_Condition_Burden,Highly correlated with an existing clinical fe...,Correlation > 0.85



Final number of features: 70

Final Feature Set:
['age', 'bmi', 'portal_logins_monthly', 'app_usage_monthly', 'messages_to_provider_90d', 'appointment_adherence_pct', 'satisfaction_score', 'days_since_last_visit', 'total_visits_12mo', 'ed_visits_12mo', 'total_cost_12mo', 'num_clinical_visits', 'avg_num_medications', 'max_num_medications', 'avg_num_chronic_conditions', 'max_num_chronic_conditions', 'has_chronic_diagnosis', 'avg_systolic_bp', 'avg_diastolic_bp', 'avg_heart_rate', 'bmi_was_missing', 'smoking_status_was_missing', 'satisfaction_score_was_missing', 'ED_Visits_Ratio', 'Cost_Per_Visit', 'Engagement_Level', 'Enrollment_Tenure_Months', 'gender_Male', 'gender_Non-binary', 'smoking_status_Former', 'smoking_status_Never', 'insurance_type_Medicare', 'insurance_type_Private', 'insurance_type_Self-Pay', 'clinic_id_10', 'clinic_id_11', 'clinic_id_12', 'clinic_id_13', 'clinic_id_14', 'clinic_id_15', 'clinic_id_16', 'clinic_id_17', 'clinic_id_18', 'clinic_id_2', 'clinic_id_3', 'clinic_i

**Leakage Note:**  
Sections 3–5 use full-dataset copies only for exploratory comparison, cleaning decisions, and feature design. The final modeling pipeline in Part 6 starts again from `merged_df`, performs the train-test split first, and fits learned preprocessing such as imputation, scaling, and encoding using the training data only.

---

## Part 6: Data Pipeline
### 6.1 Train-Test Split

**Task:** Create a proper train-test split. Use `was_readmitted_30d` as the target variable.

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import KNNImputer

print("=" * 60)
print("6.1 Leakage-Safe Train-Test Pipeline")
print("=" * 60)

# --------------------------------------------------
# 1. Split BEFORE learned preprocessing
# --------------------------------------------------
train_df, test_df = train_test_split(
    merged_df.copy(),
    test_size=0.20,
    random_state=42,
    stratify=merged_df['was_readmitted_30d']
)

y_train = train_df['was_readmitted_30d'].astype(int)
y_test = test_df['was_readmitted_30d'].astype(int)

# --------------------------------------------------
# 2. Missing indicators created before imputation
# --------------------------------------------------
indicator_cols = ['bmi', 'smoking_status', 'satisfaction_score']

for col in indicator_cols:
    train_df[f'{col}_was_missing'] = train_df[col].isna().astype(int)
    test_df[f'{col}_was_missing'] = test_df[col].isna().astype(int)

# --------------------------------------------------
# 3. Treat clear data-entry errors
# --------------------------------------------------
def mask_errors(df):
    df = df.copy()

    df.loc[(df['age'] <= 0) | (df['age'] > 120), 'age'] = np.nan
    df.loc[df['total_cost_12mo'] <= 0.01, 'total_cost_12mo'] = np.nan
    df.loc[df['total_visits_12mo'] >= 1000, 'total_visits_12mo'] = np.nan

    df.loc[
        (df['avg_systolic_bp'] < 50) |
        (df['avg_systolic_bp'] > 250),
        'avg_systolic_bp'
    ] = np.nan

    return df

train_df = mask_errors(train_df)
test_df = mask_errors(test_df)

# --------------------------------------------------
# 4. Median imputation learned from TRAIN only
# --------------------------------------------------
median_cols = [
    'age',
    'total_cost_12mo',
    'total_visits_12mo',
    'avg_systolic_bp',
    'avg_heart_rate'
]

for col in median_cols:
    train_median = train_df[col].median()

    train_df[col] = train_df[col].fillna(train_median)
    test_df[col] = test_df[col].fillna(train_median)

# --------------------------------------------------
# 5. Categorical mode imputation from TRAIN only
# --------------------------------------------------
categorical_missing_cols = [
    'smoking_status',
    'insurance_type'
]

for col in categorical_missing_cols:
    train_mode = train_df[col].mode()[0]

    train_df[col] = train_df[col].fillna(train_mode)
    test_df[col] = test_df[col].fillna(train_mode)

# --------------------------------------------------
# 6. KNN imputation fitted on TRAIN only
# --------------------------------------------------
knn_cols = [
    'bmi',
    'app_usage_monthly',
    'messages_to_provider_90d',
    'satisfaction_score',
    'avg_diastolic_bp'
]

knn_predictors = [
    col for col in train_df.select_dtypes(include=np.number).columns
    if col not in [
        'clinic_id',
        'zip_code',
        'was_readmitted_30d'
    ]
    and not col.endswith('_was_missing')
]

scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_df[knn_predictors])
test_scaled = scaler.transform(test_df[knn_predictors])

knn = KNNImputer(n_neighbors=5)

train_knn = knn.fit_transform(train_scaled)
test_knn = knn.transform(test_scaled)

train_knn = pd.DataFrame(
    scaler.inverse_transform(train_knn),
    columns=knn_predictors,
    index=train_df.index
)

test_knn = pd.DataFrame(
    scaler.inverse_transform(test_knn),
    columns=knn_predictors,
    index=test_df.index
)

for col in knn_cols:
    train_df[col] = train_knn[col]
    test_df[col] = test_knn[col]

# Count-like variables should be whole numbers
for col in ['app_usage_monthly', 'messages_to_provider_90d']:
    train_df[col] = train_df[col].round()
    test_df[col] = test_df[col].round()

# --------------------------------------------------
# 7. Feature engineering
# --------------------------------------------------
reference_date = pd.Timestamp('2025-12-31')

def engineer_features(df):
    df = df.copy()

    df['enrollment_date'] = pd.to_datetime(df['enrollment_date'])

    df['Chronic_Condition_Burden'] = (
        df['avg_num_chronic_conditions'] +
        df['has_chronic_diagnosis'].astype(int)
    )

    df['Medication_Complexity_Index'] = (
        df['avg_num_medications'] +
        df['max_num_medications']
    )

    df['ED_Visits_Ratio'] = np.where(
        df['total_visits_12mo'] > 0,
        df['ed_visits_12mo'] / df['total_visits_12mo'],
        0
    )

    df['Cost_Per_Visit'] = np.where(
        df['total_visits_12mo'] > 0,
        df['total_cost_12mo'] / df['total_visits_12mo'],
        0
    )

    df['Engagement_Level'] = (
        df['portal_logins_monthly'] +
        df['app_usage_monthly'] +
        df['messages_to_provider_90d']
    )

    df['Enrollment_Tenure_Months'] = (
        (reference_date - df['enrollment_date']).dt.days / 30.44
    ).clip(lower=0)

    return df

train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

# --------------------------------------------------
# 8. One-hot encoding fitted on TRAIN only
# --------------------------------------------------
for df in [train_df, test_df]:
    df['clinic_id'] = df['clinic_id'].astype(str)
    df['zip_code'] = df['zip_code'].astype(str)

categorical_cols = [
    'gender',
    'smoking_status',
    'insurance_type',
    'clinic_id',
    'enrollment_channel',
    'zip_code',
    'most_common_diagnosis_category'
]

encoder = OneHotEncoder(
    drop='first',
    handle_unknown='ignore',
    sparse_output=False
)

train_encoded = encoder.fit_transform(train_df[categorical_cols])
test_encoded = encoder.transform(test_df[categorical_cols])

encoded_names = encoder.get_feature_names_out(categorical_cols)

train_encoded = pd.DataFrame(
    train_encoded,
    columns=encoded_names,
    index=train_df.index
)

test_encoded = pd.DataFrame(
    test_encoded,
    columns=encoded_names,
    index=test_df.index
)

# --------------------------------------------------
# 9. Build X_train and X_test
# --------------------------------------------------
drop_cols = [
    'patient_id',
    'enrollment_date',
    'was_readmitted_30d'
] + categorical_cols

X_train = train_df.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)

# Convert booleans to 0/1
for col in X_train.select_dtypes(include='bool').columns:
    X_train[col] = X_train[col].astype(int)
    X_test[col] = X_test[col].astype(int)

X_train = pd.concat([X_train, train_encoded], axis=1)
X_test = pd.concat([X_test, test_encoded], axis=1)

# --------------------------------------------------
# 10. Feature selection learned from TRAIN only
# --------------------------------------------------

# Remove constant features
constant_cols = [
    col for col in X_train.columns
    if X_train[col].nunique() <= 1
]

X_train = X_train.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols)

# Remove exact duplicate features
duplicate_cols = []

for i, col1 in enumerate(X_train.columns):
    for col2 in X_train.columns[i + 1:]:
        if X_train[col1].equals(X_train[col2]):
            duplicate_cols.append(col2)

duplicate_cols = list(dict.fromkeys(duplicate_cols))

X_train = X_train.drop(columns=duplicate_cols)
X_test = X_test.drop(columns=duplicate_cols)

# Check high correlation using TRAIN only
corr = X_train.corr().abs()
upper = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

high_corr_pairs = []

for col in upper.columns:
    for row in upper.index:
        if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > 0.85:
            high_corr_pairs.append((row, col, upper.loc[row, col]))

# Prefer removing redundant engineered features
corr_drop = []

for f1, f2, value in high_corr_pairs:
    if 'Medication_Complexity_Index' in [f1, f2]:
        corr_drop.append('Medication_Complexity_Index')

    elif 'Chronic_Condition_Burden' in [f1, f2]:
        corr_drop.append('Chronic_Condition_Burden')

corr_drop = list(set(corr_drop))

X_train = X_train.drop(columns=corr_drop, errors='ignore')
X_test = X_test.drop(columns=corr_drop, errors='ignore')

# --------------------------------------------------
# 11. Final verification
# --------------------------------------------------
print("\nTrain/Test Shapes:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nReadmission Rates:")
print(f"Full dataset: {merged_df['was_readmitted_30d'].mean():.2%}")
print(f"Training set: {y_train.mean():.2%}")
print(f"Test set:     {y_test.mean():.2%}")

print("\nLeakage Checks:")
print("Target in X_train:", 'was_readmitted_30d' in X_train.columns)
print("patient_id in X_train:", 'patient_id' in X_train.columns)

print("\nRemaining Missing Values:")
print("X_train:", X_train.isna().sum().sum())
print("X_test :", X_test.isna().sum().sum())

print("\nTrain/Test columns identical:",
      X_train.columns.equals(X_test.columns))

print("\nFinal number of features:", X_train.shape[1])


6.1 Leakage-Safe Train-Test Pipeline

Train/Test Shapes:
X_train: (640, 70)
X_test : (160, 70)
y_train: (640,)
y_test : (160,)

Readmission Rates:
Full dataset: 39.38%
Training set: 39.38%
Test set:     39.38%

Leakage Checks:
Target in X_train: False
patient_id in X_train: False

Remaining Missing Values:
X_train: 0
X_test : 0

Train/Test columns identical: True

Final number of features: 70


### 6.2 Verification Checks

**Task:** Verify your data preparation was successful:
1. No missing values remain
2. No obvious outliers remain
3. Feature summary statistics look reasonable (mean, std, min, max)
4. Train and test sets have similar distributions

In [13]:
import pandas as pd
import numpy as np

print("=" * 60)
print("6.2 Data Verification Checks")
print("=" * 60)

results = []

# 1. Missing and infinite values
missing = X_train.isna().sum().sum() + X_test.isna().sum().sum()
infinite = np.isinf(X_train).sum().sum() + np.isinf(X_test).sum().sum()

results.append({
    "Check": "No missing values",
    "Status": "PASS" if missing == 0 and infinite == 0 else "FAIL",
    "Evidence": f"Missing values: {missing}; infinite values: {infinite}"
})

# 2. Obvious invalid outliers
invalid_age = ((X_train["age"] <= 0) | (X_train["age"] > 120)).sum() + \
              ((X_test["age"] <= 0) | (X_test["age"] > 120)).sum()

invalid_cost = (X_train["total_cost_12mo"] <= 0.01).sum() + \
               (X_test["total_cost_12mo"] <= 0.01).sum()

invalid_visits = (X_train["total_visits_12mo"] >= 1000).sum() + \
                 (X_test["total_visits_12mo"] >= 1000).sum()

invalid_bp = ((X_train["avg_systolic_bp"] < 50) |
              (X_train["avg_systolic_bp"] > 250)).sum() + \
             ((X_test["avg_systolic_bp"] < 50) |
              (X_test["avg_systolic_bp"] > 250)).sum()

total_invalid = invalid_age + invalid_cost + invalid_visits + invalid_bp

results.append({
    "Check": "Outliers addressed",
    "Status": "PASS" if total_invalid == 0 else "FAIL",
    "Evidence": f"Remaining obvious invalid values: {total_invalid}"
})

# 3. Summary statistics
key_features = [
    "age",
    "total_cost_12mo",
    "total_visits_12mo",
    "avg_systolic_bp",
    "Enrollment_Tenure_Months"
]

print("\nKey Feature Summary Statistics:")
display(X_train[key_features].describe().round(2))

results.append({
    "Check": "Reasonable distributions",
    "Status": "PASS",
    "Evidence": "Key feature ranges and summary statistics are reasonable after cleaning."
})

# 4. Train/test columns
columns_match = X_train.columns.equals(X_test.columns)

results.append({
    "Check": "Train/test columns match",
    "Status": "PASS" if columns_match else "FAIL",
    "Evidence": f"Both datasets contain {X_train.shape[1]} features."
})

# 5. Leakage check
no_leakage = (
    "patient_id" not in X_train.columns and
    "was_readmitted_30d" not in X_train.columns
)

results.append({
    "Check": "No identifier/target leakage",
    "Status": "PASS" if no_leakage else "FAIL",
    "Evidence": "patient_id and was_readmitted_30d are not included in X."
})

# 6. Train/test target balance
train_rate = y_train.mean()
test_rate = y_test.mean()

results.append({
    "Check": "Train/test balance",
    "Status": "PASS" if abs(train_rate - test_rate) < 0.01 else "FAIL",
    "Evidence": f"Train: {train_rate:.2%}; Test: {test_rate:.2%}"
})

print("\nVerification Results:")
verification_df = pd.DataFrame(results)
display(verification_df)

6.2 Data Verification Checks

Key Feature Summary Statistics:


,age,total_cost_12mo,total_visits_12mo,avg_systolic_bp,Enrollment_Tenure_Months
count,640.00,640.00,640.00,640.00,640.00
mean,57.20,8818.90,4.95,130.50,60.26
std,15.51,12894.56,2.25,12.40,33.90
min,22.00,162.15,1.00,88.00,0.07
25%,46.00,2462.08,3.00,123.00,31.23
50%,58.00,4805.90,5.00,129.67,61.27
75%,68.00,10329.88,6.00,137.64,87.71
max,92.00,149630.00,13.00,179.00,119.78



Verification Results:


,Check,Status,Evidence
0,No missing values,PASS,Missing values: 0; infinite values: 0
1,Outliers addressed,PASS,Remaining obvious invalid values: 0
2,Reasonable distributions,PASS,Key feature ranges and summary statistics are ...
3,Train/test columns match,PASS,Both datasets contain 70 features.
4,No identifier/target leakage,PASS,patient_id and was_readmitted_30d are not incl...
5,Train/test balance,PASS,Train: 39.38%; Test: 39.38%


**Verification Results:**

| Check                    | Status | Evidence                                                  |
| ------------------------ | ------ | --------------------------------------------------------- |
| No missing values        | Pass   | 0 missing and 0 infinite values                           |
| Outliers addressed       | Pass   | 0 obvious invalid values remain                           |
| Reasonable distributions | Pass   | Key feature ranges and summary statistics look reasonable |
| Train/test balance       | Pass   | Train = 39.38%, Test = 39.38%                             |


---

## Part 7: Business Summary

### 7.1 Executive Summary

**Task:** Write a brief summary (5-7 sentences) for Dr. Vasquez explaining:
- What data quality issues you found across the three systems
- How you addressed them
- What new features you created and why they matter for readmission prediction
- Confidence level that the data is now ready for the readmission model

**Executive Summary:**

*Your summary here (5-7 sentences): ClearPath Health had several data quality problems across the three systems, including missing values, incorrect ages, invalid costs, unrealistic visit counts, and impossible blood pressure values. I combined the three data sources using patient_id and kept all 800 patients in the final dataset. Missing values were handled using median, KNN, or mode imputation depending on the type of variable, while missing-value indicators were kept for important fields such as BMI and satisfaction score. I also corrected clear data-entry errors while keeping real extreme cases, such as high-cost patients. New features such as ED visit ratio, cost per visit, engagement level, and enrollment tenure were created because they may help explain a patient's risk of readmission. After feature selection, the final dataset contained 70 predictive features with no missing or infinite values. Overall, I am confident that the data is ready for the next step of building a readmission prediction model.*


### 7.2 Addressing Stakeholder Concerns

| Stakeholder                         | Their Concern                                | How Addressed                                                                                                                                     |
| ----------------------------------- | -------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------- |
| Dr. Elena Vasquez (CMO)             | 14% readmission rate, $2.3M CMS penalties    | Prepared a clean and model-ready dataset with 70 features that can be used to better identify patients at risk of readmission.                    |
| Raj Mehta (VP Health Informatics)   | Data across three separate systems           | Combined patient, clinical, and engagement data using `patient_id` and kept all 800 patients.                                                     |
| Dr. Sarah Kim (Quality Improvement) | Systematic missingness patterns              | Checked missing-value patterns by clinic and patient health status, added missing indicators, and used suitable imputation methods.               |
| Marcus Thompson (CFO)               | Suspicious billing values, data entry errors | Corrected invalid costs, impossible ages, extreme visit-count errors, and impossible blood pressure values while keeping real high-cost patients. |


---

## Part 8: Understanding Demonstration
### 8.1 In Your Own Words

**Answer these questions without AI assistance. Use your own understanding.**

**Question 1:** Explain the difference between MCAR, MAR, and MNAR missing data. For the ClearPath Health dataset, give one example of each type and explain why the type of missingness affects your choice of imputation strategy.

*Your answer (1 paragraph): Missing data can happen in three main ways. MCAR means data is missing by random chance, like a few missing vital signs with no clear pattern. MAR means the missingness is related to another variable, like BMI and smoking status being missing more often in certain clinics. MNAR means the missingness may be related to the missing value itself, like sicker or less satisfied patients being less likely to answer a survey. The type of missingness matters because simple imputation may work for MCAR, while MAR or possible MNAR cases may need related variables or missing-value indicators.*


**Question 2:** What is data leakage and why is it dangerous? Describe two specific ways data leakage could occur during data preparation for the ClearPath readmission model, and how you prevented it in your pipeline.

*Your answer (1 paragraph): Data leakage happens when information that should not be available during training gets into the model and makes the results look better than they really are. For example, leakage could happen if was_readmitted_30d was used as a predictor, or if imputation and scaling were fit on the full dataset before splitting it into training and testing sets. I prevented this by separating the target from the predictors and splitting the data first. Then I fit the preprocessing steps only on the training data and applied the same rules to the test data.*


**Question 3:** Why should you fit the imputer on training data only and then transform both train and test? What would happen if you fit the imputer on all data before splitting?

*Your answer (3-4 sentences): The imputer should be fit only on the training data because the test set is supposed to represent new, unseen patients. If the imputer uses the test data, it can learn information about those patients before the model is evaluated. This creates data leakage and can make the model appear more accurate than it would be in real use. Using the training data only keeps the evaluation fair.*

**Question 4:** ClearPath's clinical records are stored as nested JSON. Explain why `pd.json_normalize()` is needed and what would happen if you loaded the data with `pd.read_json()` alone. How does this relate to tidy data principles?

*Your answer (3-4 sentences): Clinical data can be stored in nested JSON, where information such as vital signs or diagnosis details is inside smaller objects. pd.json_normalize() flattens these nested objects into regular DataFrame columns that are easier to analyze. Using pd.read_json() alone may leave some nested information as dictionaries inside cells, which is harder to work with. This follows tidy data principles because each variable should have its own column and each observation should have its own row.*


---

## Part 9: Critical Analysis
### 9.1 Spot the Error

The following code contains **4 errors** related to data preparation best practices. Identify each error, explain why it's a problem, and provide the correction.

```python
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Merge patient data
df = pd.merge(patients, clinical, on='patient_id', how='inner')       # Line A

# Drop rows with missing satisfaction scores
df = df.dropna(subset=['satisfaction_score'])                          # Line B

# Impute remaining missing values on full dataset
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df.select_dtypes(include='number')),
                          columns=df.select_dtypes(include='number').columns)

# Now split
X_train, X_test, y_train, y_test = train_test_split(
    df_imputed, y, test_size=0.2, random_state=42
)                                                                      # Line C

# Impute test data separately
test_imputer = SimpleImputer(strategy='median')
X_test_imputed = test_imputer.fit_transform(X_test)                    # Line D
```

### Error 1

* **Line(s):** Line A
* **The error:** The code directly joins `patients` with visit-level `clinical` data using an inner join.
* **Why it's a problem:** One patient can have multiple clinical visits, so this can create duplicate patient rows. An inner join can also remove patients who do not have matching records.
* **Correction:** Aggregate the clinical data to one row per `patient_id` first, then use a left join to preserve all patients.

### Error 2

* **Line(s):** Line B
* **The error:** Rows with missing `satisfaction_score` are deleted.
* **Why it's a problem:** The missing satisfaction scores may follow a systematic pattern, so deleting these patients could bias the dataset and remove higher-risk patients.
* **Correction:** Keep the patients, create a missing-value indicator if useful, and impute the missing scores with an appropriate method.

### Error 3

* **Line(s):** Line C and the imputation step before it
* **The error:** The imputer is fit on the full dataset before the train-test split.
* **Why it's a problem:** This causes data leakage because information from the test set is used during preprocessing.
* **Correction:** Split the data first, fit the imputer only on the training data, then use the same fitted imputer to transform both the training and test data.

### Error 4

* **Line(s):** Line D
* **The error:** A new imputer is fit separately on the test set.
* **Why it's a problem:** The test set should remain unseen and should use the same preprocessing rules learned from the training data.
* **Correction:** Use the imputer already fit on `X_train` and call only `transform()` on `X_test`.


### 9.2 Limitations of Your Approach

**Task:** Identify 2-3 limitations of your data preparation approach. What could go wrong? What assumptions did you make?

**Limitations:**

1. The missingness type cannot always be proven from the available data. For example, the satisfaction-score pattern may be MNAR, but the evidence was mixed, so the assumption could still be wrong.
2. Imputation changes the original data. KNN and median imputation can reduce variation or create values that are not exactly what the patient originally had.
3. Clinical visits were summarized into one row per patient, so some visit timing and sequence information may have been lost. Also, the patient-level target was defined as whether a patient had any 30-day readmission across their recorded visits, so the 39.38% analytical target rate is not directly comparable to ClearPath's 14% organization-level readmission rate.

---

## Part 10: AI Usage Report
### 10.1 Prompt Evolution Log

Document how your prompts improved throughout the assignment.

| Section                | Initial Prompt                                                                                      | Result                                                                                                       | Refined Prompt                                                                                                          | Improvement                                                                                            |
| ---------------------- | --------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------ |
| Data Loading & Merging | “Complete Section 2.2 by aggregating clinical to one row per patient and merge the three datasets.” | Correct merge, but the response included long comments and unnecessary code.                                 | “Trim the comments, remove unused code, keep the merge logic unchanged, and make the explanation concise.”              | Produced cleaner code, preserved all 800 patients, and clearly explained the one-to-many relationship. |
| Missing Values         | “Analyze missing values, compare old vs newer clinics, and test satisfaction-score missingness.”    | AI initially made strong MAR/MNAR claims and used arbitrary thresholds.                                      | “Do not automatically classify MAR/MNAR, remove arbitrary thresholds, and use ‘unclear’ when evidence is insufficient.” | Made the missingness conclusions more careful and evidence-based.                                      |
| Outliers               | “Use IQR and identify data errors versus genuine extreme values.”                                   | AI generated very long code with repeated rules.                                                             | “Keep only IQR analysis and key business-rule checks for age, cost, visits, and BP.”                                    | Shortened the code while still separating errors from legitimate high-cost and clinical extremes.      |
| Feature Engineering    | “Create at least 5 useful features for readmission prediction.”                                     | AI created useful features, but some formulas had issues such as negative tenure and duplicated information. | “Avoid negative tenure, avoid unnecessary +1 denominators, and check for redundant engineered features.”                | Produced more interpretable features and removed highly correlated engineered features later.          |


### 10.2 Most Effective Prompts

List 2-3 prompts that worked particularly well and explain why.

1. **Prompt:** “Do not automatically label MAR or MNAR before examining the results, remove arbitrary thresholds, and use ‘unclear’ when evidence is insufficient.” **Why it worked:** It stopped the AI from making conclusions before checking the actual data and made the missing-value analysis more accurate.

2. **Prompt:** “Create a leakage-safe train-test pipeline, split first, then fit imputation, scaling, and encoding only on the training data.” **Why it worked:** It clearly told the AI the correct order of preprocessing and prevented information from the test set from leaking into training.

3. **Prompt:**“Separate data-entry errors from genuine extreme values and keep legitimate high-cost and high-BP patients.” **Why it worked:** It prevented the AI from automatically treating every statistical outlier as a bad value.

### 10.3 AI Limitations Encountered

**Where did AI struggle or give incorrect output?**
AI sometimes produced code that was much longer than needed, used arbitrary thresholds, and made missingness classifications too confidently. It also initially treated some numeric IDs like normal numeric features and created some redundant engineered features.

**What did you have to figure out yourself?**
I had to decide which missingness conclusions were actually supported by the data, which outliers were real errors versus valid clinical extremes, and which engineered features should be kept or removed.

**What prior knowledge was essential for verifying AI's output?**
Understanding MCAR/MAR/MNAR, data leakage, train-test preprocessing, one-to-many joins, outlier treatment, and feature correlation was important for checking whether the AI output was reasonable.


### 10.4 Confidence Self-Assessment

Rate your understanding (1=Low, 5=High):

| Concept                             | Rating | Explanation                                                                                                                     |
| ----------------------------------- | -----: | ------------------------------------------------------------------------------------------------------------------------------- |
| Missing value types (MCAR/MAR/MNAR) |      4 | I understand the main differences and how observed patterns help with classification, but MNAR can still be difficult to prove. |
| Imputation strategies               |      4 | I understand when median, mode, KNN, and missing indicators can be useful.                                                      |
| Outlier detection and treatment     |      4 | I can use IQR and business rules to separate errors from real extreme values.                                                   |
| Feature engineering                 |      4 | I can create useful features and check whether they add new information.                                                        |
| Data leakage prevention             |      5 | I understand why preprocessing must be learned from training data only.                                                         |
| Merging multiple data sources       |      5 | I understand join keys, one-to-one vs. one-to-many relationships, and why clinical data needed aggregation.                     |


---

## Video Walkthrough Instructions

**Record a 3-5 minute video covering:**

1. **Problem Context (30-60 sec):** Briefly explain ClearPath Health's data challenge and the readmission prediction goal
2. **Key Decisions (90-120 sec):** Walk through 2-3 important decisions you made:
   - How did you merge the three data sources and why?
   - How did you handle a specific missing value pattern, and how did you decide which type it was?
   - How did you distinguish data errors from genuine extreme values?
3. **Verification (30-60 sec):** How do you know your data is ML-ready?
4. **Limitations (30 sec):** One limitation ClearPath stakeholders should understand

**Upload to Canvas with your notebook submission.**

---

## Submission Checklist

Before submitting, verify:

- [ ] All code cells execute without errors (Kernel > Restart and Run All)
- [ ] Problem Framing section completed with business understanding
- [ ] All three data sources loaded and merged with rationale
- [ ] Missing value analysis includes pattern identification (MCAR/MAR/MNAR)
- [ ] Imputation strategies justified with clinical rationale
- [ ] At least 5 engineered features with documentation
- [ ] Outlier treatment decisions documented (errors vs. genuine extremes)
- [ ] Train-test split with no data leakage
- [ ] Verification checks completed
- [ ] All 4 errors identified in Spot the Error section
- [ ] Understanding questions answered in your own words
- [ ] AI Usage Report section fully completed
- [ ] Video walkthrough uploaded

**File naming:** `LastName_FirstName_Week03.ipynb`